---
title: "Practice Activity 3-2: Distances Between Observations"
author: "Shiqi Wu"
format:
  html:
    embed-resources: true
    code-fold: false
---

# Distances Between Observations

Read this notebook from top to bottom and fill in the code as you go. Work together and discuss with other students in the class. Try to resolve any errors on your own first, but don't get stuck; ask for help!

In addition to writing and running code, be sure to examine any output and interpret the results before moving on.

For many of these questions, there are several approaches, and there is no single right answer. You should try a few different things and compare with your classmates.

We will use `scikit-learn` extensively later, but for this activity you might want to stick with `pandas`.


In [251]:
import pandas as pd
import numpy as np

## Ames - Recommending Similar Homes

1\. Suppose that you really like house 0 in the Ames housing data set, but it is too expensive. Find cheaper homes that are similar to it --- in terms of living area, number of bedrooms, number of bathrooms --- by calculating distances from house 0. You might want to try different distance metrics and different scaling methods; how sensitive are your results to these choices?

Be sure to actually look at the profiles of the homes that your algorithm picked out as most similar (based on these 3 variables). Do they make sense?

_Think:_ If the goal is to find a "good deal" on a similar house, should sale price be included as a variable in your distance metric?

In [252]:
df_housing = pd.read_csv("https://raw.githubusercontent.com/kevindavisross/data301/main/data/AmesHousing.txt",sep="\t")

In [253]:
df_housing.head()

,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,...,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice
0,1,526301100,20,RL,141.0,31770,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,NaN,0,5,2010,WD,Normal,215000
1,2,526350040,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,...,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal,105000
2,3,526351010,20,RL,81.0,14267,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal,172000
3,4,526353030,20,RL,93.0,11160,Pave,NaN,Reg,Lvl,...,0,NaN,NaN,NaN,0,4,2010,WD,Normal,244000
4,5,527105010,60,RL,74.0,13830,Pave,NaN,IR1,Lvl,...,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal,189900


In [254]:
df_housing["Bathrooms"] = (df_housing["Full Bath"]+ df_housing["Bsmt Full Bath"]+ 0.5 * (df_housing["Half Bath"] + df_housing["Bsmt Half Bath"]))

In [255]:
df_housing.loc[0, ["Gr Liv Area", "Bedroom AbvGr", "Bathrooms", "SalePrice"]]

Gr Liv Area        1656.0
Bedroom AbvGr         3.0
Bathrooms             2.0
SalePrice        215000.0
Name: 0, dtype: float64

In [256]:
df_housing_quant = df_housing[["Gr Liv Area", "Bedroom AbvGr", "Bathrooms"]]

df_housing_quant.head()

,Gr Liv Area,Bedroom AbvGr,Bathrooms
0,1656,3,2.0
1,896,2,1.0
2,1329,3,1.5
3,2110,3,3.5
4,1629,3,2.5


In [257]:
df_housing_st = ((df_housing_quant - df_housing_quant.mean())/ df_housing_quant.std())

df_housing_st.head()

,Gr Liv Area,Bedroom AbvGr,Bathrooms
0,0.309212,0.176064,-0.269988
1,-1.194223,-1.032058,-1.509056
2,-0.337661,0.176064,-0.889522
3,1.207317,0.176064,1.588613
4,0.255801,0.176064,0.349546


In [258]:
house_0 = df_housing_st.loc[0]

In [259]:
house_0

Gr Liv Area      0.309212
Bedroom AbvGr    0.176064
Bathrooms       -0.269988
Name: 0, dtype: float64

In [260]:
distance_euclidean = np.sqrt(((df_housing_st - house_0) ** 2).sum(axis=1))

In [261]:
distance_euclidean.head()

0    0.000000
1    2.292415
2    0.895693
3    2.064217
4    0.621832
dtype: float64

In [262]:
df_housing["distance_euclidean"] = distance_euclidean

In [263]:
df_housing[["SalePrice", "distance_euclidean"]].head()

,SalePrice,distance_euclidean
0,215000,0.000000
1,105000,2.292415
2,172000,0.895693
3,244000,2.064217
4,189900,0.621832


In [264]:
cheaper_homes = df_housing[df_housing["SalePrice"] < df_housing.loc[0, "SalePrice"]]

In [265]:
cheaper_homes[["SalePrice", "distance_euclidean"]].head()

,SalePrice,distance_euclidean
1,105000,2.292415
2,172000,0.895693
4,189900,0.621832
5,195500,0.628016
6,213500,1.841351


In [266]:
cheaper_homes.sort_values("distance_euclidean")[["Gr Liv Area", "Bedroom AbvGr", "Bathrooms", "SalePrice", "distance_euclidean"]].head()

,Gr Liv Area,Bedroom AbvGr,Bathrooms,SalePrice,distance_euclidean
731,1654,3,2.0,137000,0.003956
1226,1661,3,2.0,165500,0.009891
1994,1650,3,2.0,144100,0.011869
2673,1664,3,2.0,159000,0.015826
1940,1647,3,2.0,153000,0.017804


In [267]:
distance_manhattan = (df_housing_st - house_0).abs().sum(axis=1)

distance_manhattan.head()

0    0.000000
1    3.950625
2    1.266407
3    2.756706
4    0.672945
dtype: float64

In [268]:
cheaper_homes = cheaper_homes.assign(distance_manhattan=distance_manhattan)

cheaper_homes[["SalePrice", "distance_euclidean", "distance_manhattan"]].head()

,SalePrice,distance_euclidean,distance_manhattan
1,105000,2.292415,3.950625
2,172000,0.895693,1.266407
4,189900,0.621832,0.672945
5,195500,0.628016,0.722400
6,213500,1.841351,3.076258


In [269]:
cheaper_homes.sort_values("distance_manhattan")[["Gr Liv Area", "Bedroom AbvGr", "Bathrooms", "SalePrice", "distance_manhattan"]].head()

,Gr Liv Area,Bedroom AbvGr,Bathrooms,SalePrice,distance_manhattan
731,1654,3,2.0,137000,0.003956
1226,1661,3,2.0,165500,0.009891
1994,1650,3,2.0,144100,0.011869
2673,1664,3,2.0,159000,0.015826
1940,1647,3,2.0,153000,0.017804


In [270]:
df_housing_mm = ((df_housing_quant - df_housing_quant.min())/ (df_housing_quant.max() - df_housing_quant.min()))

df_housing_mm.head()

,Gr Liv Area,Bedroom AbvGr,Bathrooms
0,0.249058,0.375,0.166667
1,0.105878,0.250,0.000000
2,0.187453,0.375,0.083333
3,0.334589,0.375,0.416667
4,0.243971,0.375,0.250000


In [271]:
house_0_mm = df_housing_mm.loc[0]

distance_euclidean_mm = np.sqrt(((df_housing_mm - house_0_mm) ** 2).sum(axis=1))

In [272]:
distance_euclidean_mm.head()

0    0.000000
1    0.252791
2    0.103632
3    0.264226
4    0.083488
dtype: float64

In [273]:
cheaper_homes = cheaper_homes.assign(distance_euclidean_mm=distance_euclidean_mm)

cheaper_homes[["SalePrice", "distance_euclidean", "distance_euclidean_mm"]].head()

,SalePrice,distance_euclidean,distance_euclidean_mm
1,105000,2.292415,0.252791
2,172000,0.895693,0.103632
4,189900,0.621832,0.083488
5,195500,0.628016,0.083907
6,213500,1.841351,0.216776


In [274]:
cheaper_homes.sort_values("distance_euclidean_mm")[["Gr Liv Area", "Bedroom AbvGr", "Bathrooms", "SalePrice", "distance_euclidean_mm"]].head()

,Gr Liv Area,Bedroom AbvGr,Bathrooms,SalePrice,distance_euclidean_mm
731,1654,3,2.0,137000,0.000377
1226,1661,3,2.0,165500,0.000942
1994,1650,3,2.0,144100,0.001130
2673,1664,3,2.0,159000,0.001507
1940,1647,3,2.0,153000,0.001696


Answer: I used standardized living area, bedrooms, and bathrooms to calculate Euclidean distance. I only kept homes cheaper than house 0 which is 215,000. The closest five were 731, 1226, 1994, 2673, and 1940.

They all have 3 bedrooms and 2 bathrooms, the same as house 0. Their living areas are between 1,647 and 1,664 square feet, close to house 0's 1,656. House 731 is the closest and costs 137,000. Based on these features, the results make sense.

I also tried Manhattan distance with standardized data and Euclidean distance with min-max scaling. The top five homes and their order stayed the same.

I did not use price to calculate distance because I wanted similar homes, not similar prices. I used price only to find cheaper homes.

2\. Continuing part 1. Suppose that you really like house 0 in the data set, but it is too expensive. Find cheaper homes that are similar to it --- in terms of living area, number of bedrooms, number of bathrooms, **and House Style** --- by calculating distances from house 0. You might want to try different distance metrics and different scaling methods; how sensitive are your results to these choices?

Be sure to actually look at the profiles of the homes that your algorithm picked out as most similar. Do they make sense?

In [275]:
df_housing.loc[0, ["Gr Liv Area", "Bedroom AbvGr", "Bathrooms", "House Style", "SalePrice"]]

Gr Liv Area        1656
Bedroom AbvGr         3
Bathrooms           2.0
House Style      1Story
SalePrice        215000
Name: 0, dtype: object

In [276]:
house_style_dummies = pd.get_dummies(df_housing[["House Style"]], dtype=int)

In [277]:
house_style_dummies.head()

,House Style_1.5Fin,House Style_1.5Unf,House Style_1Story,House Style_2.5Fin,House Style_2.5Unf,House Style_2Story,House Style_SFoyer,House Style_SLvl
0,0,0,1,0,0,0,0,0
1,0,0,1,0,0,0,0,0
2,0,0,1,0,0,0,0,0
3,0,0,1,0,0,0,0,0
4,0,0,0,0,0,1,0,0


In [278]:
df_housing_mixed = pd.concat([df_housing_st, house_style_dummies], axis=1)

In [279]:
df_housing_mixed.head()

,Gr Liv Area,Bedroom AbvGr,Bathrooms,House Style_1.5Fin,House Style_1.5Unf,House Style_1Story,House Style_2.5Fin,House Style_2.5Unf,House Style_2Story,House Style_SFoyer,House Style_SLvl
0,0.309212,0.176064,-0.269988,0,0,1,0,0,0,0,0
1,-1.194223,-1.032058,-1.509056,0,0,1,0,0,0,0,0
2,-0.337661,0.176064,-0.889522,0,0,1,0,0,0,0,0
3,1.207317,0.176064,1.588613,0,0,1,0,0,0,0,0
4,0.255801,0.176064,0.349546,0,0,0,0,0,1,0,0


In [280]:
house_0_mixed = df_housing_mixed.loc[0]

house_0_mixed

Gr Liv Area           0.309212
Bedroom AbvGr         0.176064
Bathrooms            -0.269988
House Style_1.5Fin    0.000000
House Style_1.5Unf    0.000000
House Style_1Story    1.000000
House Style_2.5Fin    0.000000
House Style_2.5Unf    0.000000
House Style_2Story    0.000000
House Style_SFoyer    0.000000
House Style_SLvl      0.000000
Name: 0, dtype: float64

In [281]:
distance_euclidean_mixed = np.sqrt(((df_housing_mixed - house_0_mixed) ** 2).sum(axis=1))

distance_euclidean_mixed.head()

0    0.000000
1    2.292415
2    0.895693
3    2.064217
4    1.544887
dtype: float64

In [282]:
cheaper_homes = cheaper_homes.assign(distance_euclidean_mixed=distance_euclidean_mixed)

cheaper_homes[["SalePrice", "House Style", "distance_euclidean_mixed"]].head()

,SalePrice,House Style,distance_euclidean_mixed
1,105000,1Story,2.292415
2,172000,1Story,0.895693
4,189900,2Story,1.544887
5,195500,2Story,1.547386
6,213500,1Story,1.841351


In [283]:
cheaper_homes.sort_values("distance_euclidean_mixed")[["Gr Liv Area", "Bedroom AbvGr", "Bathrooms", "House Style", "SalePrice", "distance_euclidean_mixed"]].head()

,Gr Liv Area,Bedroom AbvGr,Bathrooms,House Style,SalePrice,distance_euclidean_mixed
1940,1647,3,2.0,1Story,153000,0.017804
618,1644,3,2.0,1Story,167000,0.023738
375,1671,3,2.0,1Story,200000,0.029673
209,1675,3,2.0,1Story,173000,0.037586
1185,1682,3,2.0,1Story,174000,0.051433


In [284]:
distance_manhattan_mixed = (df_housing_mixed - house_0_mixed).abs().sum(axis=1)

distance_manhattan_mixed.head()

0    0.000000
1    3.950625
2    1.266407
3    2.756706
4    2.672945
dtype: float64

In [285]:
cheaper_homes = cheaper_homes.assign(distance_manhattan_mixed=distance_manhattan_mixed)

cheaper_homes[["SalePrice", "House Style", "distance_euclidean_mixed", "distance_manhattan_mixed"]].head()

,SalePrice,House Style,distance_euclidean_mixed,distance_manhattan_mixed
1,105000,1Story,2.292415,3.950625
2,172000,1Story,0.895693,1.266407
4,189900,2Story,1.544887,2.672945
5,195500,2Story,1.547386,2.722400
6,213500,1Story,1.841351,3.076258


In [286]:
cheaper_homes.sort_values("distance_manhattan_mixed")[["Gr Liv Area", "Bedroom AbvGr", "Bathrooms", "House Style", "SalePrice", "distance_manhattan_mixed"]].head()

,Gr Liv Area,Bedroom AbvGr,Bathrooms,House Style,SalePrice,distance_manhattan_mixed
1940,1647,3,2.0,1Story,153000,0.017804
618,1644,3,2.0,1Story,167000,0.023738
375,1671,3,2.0,1Story,200000,0.029673
209,1675,3,2.0,1Story,173000,0.037586
1185,1682,3,2.0,1Story,174000,0.051433


In [287]:
df_housing_mixed_mm = pd.concat([df_housing_mm, house_style_dummies], axis=1)

df_housing_mixed_mm.head()

,Gr Liv Area,Bedroom AbvGr,Bathrooms,House Style_1.5Fin,House Style_1.5Unf,House Style_1Story,House Style_2.5Fin,House Style_2.5Unf,House Style_2Story,House Style_SFoyer,House Style_SLvl
0,0.249058,0.375,0.166667,0,0,1,0,0,0,0,0
1,0.105878,0.250,0.000000,0,0,1,0,0,0,0,0
2,0.187453,0.375,0.083333,0,0,1,0,0,0,0,0
3,0.334589,0.375,0.416667,0,0,1,0,0,0,0,0
4,0.243971,0.375,0.250000,0,0,0,0,0,1,0,0


In [288]:
house_0_mixed_mm = df_housing_mixed_mm.loc[0]

house_0_mixed_mm

Gr Liv Area           0.249058
Bedroom AbvGr         0.375000
Bathrooms             0.166667
House Style_1.5Fin    0.000000
House Style_1.5Unf    0.000000
House Style_1Story    1.000000
House Style_2.5Fin    0.000000
House Style_2.5Unf    0.000000
House Style_2Story    0.000000
House Style_SFoyer    0.000000
House Style_SLvl      0.000000
Name: 0, dtype: float64

In [289]:
distance_euclidean_mixed_mm = np.sqrt(((df_housing_mixed_mm - house_0_mixed_mm) ** 2).sum(axis=1))

distance_euclidean_mixed_mm.head()

0    0.000000
1    0.252791
2    0.103632
3    0.264226
4    1.416676
dtype: float64

In [290]:
cheaper_homes = cheaper_homes.assign(distance_euclidean_mixed_mm=distance_euclidean_mixed_mm)

cheaper_homes[["SalePrice", "House Style", "distance_euclidean_mixed", "distance_euclidean_mixed_mm"]].head()

,SalePrice,House Style,distance_euclidean_mixed,distance_euclidean_mixed_mm
1,105000,1Story,2.292415,0.252791
2,172000,1Story,0.895693,0.103632
4,189900,2Story,1.544887,1.416676
5,195500,2Story,1.547386,1.416701
6,213500,1Story,1.841351,0.216776


In [291]:
cheaper_homes.sort_values("distance_euclidean_mixed_mm")[["Gr Liv Area", "Bedroom AbvGr", "Bathrooms", "House Style","SalePrice", "distance_euclidean_mixed_mm"]].head()

,Gr Liv Area,Bedroom AbvGr,Bathrooms,House Style,SalePrice,distance_euclidean_mixed_mm
1940,1647,3,2.0,1Story,153000,0.001696
618,1644,3,2.0,1Story,167000,0.002261
375,1671,3,2.0,1Story,200000,0.002826
209,1675,3,2.0,1Story,173000,0.003580
876,1630,3,2.0,1Story,213000,0.004898


Answer: I standardized living area, bedrooms, and bathrooms, and converted House Style into dummy variables without scaling them. I used Euclidean distance and only kept homes cheaper than house 0.

The closest five were 1940, 618, 375, 209, and 1185. They all have 3 bedrooms, 2 bathrooms, and the same 1Story style as house 0. Their living areas are also close. House 1940 is the closest and costs 153,000, compared with 215,000 for house 0. Based on these features, the results make sense.

Manhattan distance gave the same top five. With min-max scaling, the first four stayed the same, but house 876 appeared fifth instead of 1185. These two homes have equal distances because both differ from house 0 by 26 square feet and match its other features. Overall, the recommendations were similar across the methods I tried.

3\. Continuing parts 1 and 2. Suppose that you really like house 0 in the data set, but it is too expensive. Find cheaper homes that are similar to it, by calculating distances. You can **choose the variables to include, but include both quantitative and categorical variables**. Be sure to actually look at the profiles of the homes that your algorithm picked out as most similar. Do they make sense?

You might want to try different distance metrics and different scaling methods; how sensitive are your results to these choices?

_Hint:_ There are many variables in the data set. Do not attempt to compute distance based on all the variables! You will want to pare down the number of variables, but be sure to include a mixture of categorical and quantitative variables. Refer to the [data documentation](https://ww2.amstat.org/publications/jse/v19n3/decock/DataDocumentation.txt) for information about the variables.


In [292]:
df_housing.loc[0, ["Gr Liv Area", "Bedroom AbvGr", "Bathrooms","House Style", "Neighborhood", "SalePrice"]]

Gr Liv Area        1656
Bedroom AbvGr         3
Bathrooms           2.0
House Style      1Story
Neighborhood      NAmes
SalePrice        215000
Name: 0, dtype: object

In [293]:
housing_categories = pd.get_dummies(df_housing[["House Style", "Neighborhood"]],dtype=int)

housing_categories.head()

,House Style_1.5Fin,House Style_1.5Unf,House Style_1Story,House Style_2.5Fin,House Style_2.5Unf,House Style_2Story,House Style_SFoyer,House Style_SLvl,Neighborhood_Blmngtn,Neighborhood_Blueste,...,Neighborhood_NoRidge,Neighborhood_NridgHt,Neighborhood_OldTown,Neighborhood_SWISU,Neighborhood_Sawyer,Neighborhood_SawyerW,Neighborhood_Somerst,Neighborhood_StoneBr,Neighborhood_Timber,Neighborhood_Veenker
0,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [294]:
df_housing_selected = pd.concat([df_housing_st, housing_categories], axis=1)

df_housing_selected.head()

,Gr Liv Area,Bedroom AbvGr,Bathrooms,House Style_1.5Fin,House Style_1.5Unf,House Style_1Story,House Style_2.5Fin,House Style_2.5Unf,House Style_2Story,House Style_SFoyer,...,Neighborhood_NoRidge,Neighborhood_NridgHt,Neighborhood_OldTown,Neighborhood_SWISU,Neighborhood_Sawyer,Neighborhood_SawyerW,Neighborhood_Somerst,Neighborhood_StoneBr,Neighborhood_Timber,Neighborhood_Veenker
0,0.309212,0.176064,-0.269988,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,-1.194223,-1.032058,-1.509056,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,-0.337661,0.176064,-0.889522,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,1.207317,0.176064,1.588613,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0.255801,0.176064,0.349546,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0


In [295]:
house_0_selected = df_housing_selected.loc[0]

house_0_selected

Gr Liv Area             0.309212
Bedroom AbvGr           0.176064
Bathrooms              -0.269988
House Style_1.5Fin      0.000000
House Style_1.5Unf      0.000000
House Style_1Story      1.000000
House Style_2.5Fin      0.000000
House Style_2.5Unf      0.000000
House Style_2Story      0.000000
House Style_SFoyer      0.000000
House Style_SLvl        0.000000
Neighborhood_Blmngtn    0.000000
Neighborhood_Blueste    0.000000
Neighborhood_BrDale     0.000000
Neighborhood_BrkSide    0.000000
Neighborhood_ClearCr    0.000000
Neighborhood_CollgCr    0.000000
Neighborhood_Crawfor    0.000000
Neighborhood_Edwards    0.000000
Neighborhood_Gilbert    0.000000
Neighborhood_Greens     0.000000
Neighborhood_GrnHill    0.000000
Neighborhood_IDOTRR     0.000000
Neighborhood_Landmrk    0.000000
Neighborhood_MeadowV    0.000000
Neighborhood_Mitchel    0.000000
Neighborhood_NAmes      1.000000
Neighborhood_NPkVill    0.000000
Neighborhood_NWAmes     0.000000
Neighborhood_NoRidge    0.000000
Neighborho

In [296]:
distance_euclidean_selected = np.sqrt(((df_housing_selected - house_0_selected) ** 2).sum(axis=1))

distance_euclidean_selected.head()

0    0.000000
1    2.292415
2    0.895693
3    2.064217
4    2.094439
dtype: float64

In [297]:
cheaper_homes = cheaper_homes.assign(distance_euclidean_selected=distance_euclidean_selected)

cheaper_homes[["SalePrice", "Neighborhood", "distance_euclidean_selected"]].head()

,SalePrice,Neighborhood,distance_euclidean_selected
1,105000,NAmes,2.292415
2,172000,NAmes,0.895693
4,189900,Gilbert,2.094439
5,195500,Gilbert,2.096283
6,213500,StoneBr,2.321761


In [298]:
cheaper_homes.sort_values("distance_euclidean_selected")[["Gr Liv Area", "Bedroom AbvGr", "Bathrooms", "House Style", "Neighborhood", "SalePrice", "distance_euclidean_selected"]].head()

,Gr Liv Area,Bedroom AbvGr,Bathrooms,House Style,Neighborhood,SalePrice,distance_euclidean_selected
1940,1647,3,2.0,1Story,NAmes,153000,0.017804
618,1644,3,2.0,1Story,NAmes,167000,0.023738
1611,1587,3,2.0,1Story,NAmes,167300,0.136496
1240,1570,3,2.0,1Story,NAmes,166800,0.170126
1248,1790,3,2.0,1Story,NAmes,180000,0.265079


In [299]:
distance_manhattan_selected = (df_housing_selected - house_0_selected).abs().sum(axis=1)

distance_manhattan_selected.head()

0    0.000000
1    3.950625
2    1.266407
3    2.756706
4    4.672945
dtype: float64

In [300]:
cheaper_homes = cheaper_homes.assign(distance_manhattan_selected=distance_manhattan_selected)

cheaper_homes[["SalePrice", "distance_euclidean_selected", "distance_manhattan_selected"]].head()

,SalePrice,distance_euclidean_selected,distance_manhattan_selected
1,105000,2.292415,3.950625
2,172000,0.895693,1.266407
4,189900,2.094439,4.672945
5,195500,2.096283,4.722400
6,213500,2.321761,5.076258


In [301]:
cheaper_homes.sort_values("distance_manhattan_selected")[["Gr Liv Area", "Bedroom AbvGr", "Bathrooms", "House Style", "Neighborhood", "SalePrice", "distance_manhattan_selected"]].head()

,Gr Liv Area,Bedroom AbvGr,Bathrooms,House Style,Neighborhood,SalePrice,distance_manhattan_selected
1940,1647,3,2.0,1Story,NAmes,153000,0.017804
618,1644,3,2.0,1Story,NAmes,167000,0.023738
1611,1587,3,2.0,1Story,NAmes,167300,0.136496
1240,1570,3,2.0,1Story,NAmes,166800,0.170126
1248,1790,3,2.0,1Story,NAmes,180000,0.265079


In [302]:
df_housing_selected_mm = pd.concat([df_housing_mm, housing_categories], axis=1)

df_housing_selected_mm.head()

,Gr Liv Area,Bedroom AbvGr,Bathrooms,House Style_1.5Fin,House Style_1.5Unf,House Style_1Story,House Style_2.5Fin,House Style_2.5Unf,House Style_2Story,House Style_SFoyer,...,Neighborhood_NoRidge,Neighborhood_NridgHt,Neighborhood_OldTown,Neighborhood_SWISU,Neighborhood_Sawyer,Neighborhood_SawyerW,Neighborhood_Somerst,Neighborhood_StoneBr,Neighborhood_Timber,Neighborhood_Veenker
0,0.249058,0.375,0.166667,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0.105878,0.250,0.000000,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0.187453,0.375,0.083333,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0.334589,0.375,0.416667,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0.243971,0.375,0.250000,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0


In [303]:
house_0_selected_mm = df_housing_selected_mm.loc[0]

house_0_selected_mm

Gr Liv Area             0.249058
Bedroom AbvGr           0.375000
Bathrooms               0.166667
House Style_1.5Fin      0.000000
House Style_1.5Unf      0.000000
House Style_1Story      1.000000
House Style_2.5Fin      0.000000
House Style_2.5Unf      0.000000
House Style_2Story      0.000000
House Style_SFoyer      0.000000
House Style_SLvl        0.000000
Neighborhood_Blmngtn    0.000000
Neighborhood_Blueste    0.000000
Neighborhood_BrDale     0.000000
Neighborhood_BrkSide    0.000000
Neighborhood_ClearCr    0.000000
Neighborhood_CollgCr    0.000000
Neighborhood_Crawfor    0.000000
Neighborhood_Edwards    0.000000
Neighborhood_Gilbert    0.000000
Neighborhood_Greens     0.000000
Neighborhood_GrnHill    0.000000
Neighborhood_IDOTRR     0.000000
Neighborhood_Landmrk    0.000000
Neighborhood_MeadowV    0.000000
Neighborhood_Mitchel    0.000000
Neighborhood_NAmes      1.000000
Neighborhood_NPkVill    0.000000
Neighborhood_NWAmes     0.000000
Neighborhood_NoRidge    0.000000
Neighborho

In [304]:
distance_euclidean_selected_mm = np.sqrt(((df_housing_selected_mm - house_0_selected_mm) ** 2).sum(axis=1))

distance_euclidean_selected_mm.head()

0    0.000000
1    0.252791
2    0.103632
3    0.264226
4    2.001742
dtype: float64

In [305]:
cheaper_homes = cheaper_homes.assign(distance_euclidean_selected_mm=distance_euclidean_selected_mm)

cheaper_homes[["SalePrice", "distance_euclidean_selected", "distance_euclidean_selected_mm"]].head()

,SalePrice,distance_euclidean_selected,distance_euclidean_selected_mm
1,105000,2.292415,0.252791
2,172000,0.895693,0.103632
4,189900,2.094439,2.001742
5,195500,2.096283,2.001759
6,213500,2.321761,1.430731


In [306]:
cheaper_homes.sort_values("distance_euclidean_selected_mm")[["Gr Liv Area", "Bedroom AbvGr", "Bathrooms", "House Style", "Neighborhood", "SalePrice", "distance_euclidean_selected_mm"]].head()

,Gr Liv Area,Bedroom AbvGr,Bathrooms,House Style,Neighborhood,SalePrice,distance_euclidean_selected_mm
1940,1647,3,2.0,1Story,NAmes,153000,0.001696
618,1644,3,2.0,1Story,NAmes,167000,0.002261
1611,1587,3,2.0,1Story,NAmes,167300,0.012999
1240,1570,3,2.0,1Story,NAmes,166800,0.016202
1248,1790,3,2.0,1Story,NAmes,180000,0.025245


Answer: I used living area, bedrooms, bathrooms, House Style, and Neighborhood. I standardized the three numerical variables and converted the two categorical variables into dummy variables without scaling them. I calculated Euclidean distance and only kept homes cheaper than house 0.

The closest five were 1940, 618, 1611, 1240, and 1248. They all have 3 bedrooms, 2 bathrooms, a 1Story style, and are in NAmes, just like house 0. Their living areas range from 1,570 to 1,790 square feet, compared with 1,656 for house 0. Their prices range from 153,000 to 180,000, below house 0's 215,000. Based on these features, the results make sense.

Adding Neighborhood changed some of the recommendations from part 2. However, using Manhattan distance or min-max scaling gave the same top five in the same order. The recommendations were stable across the methods I tried.

## Colleges similar to Cal Poly

We'll use data from the [College Scorecard data](https://collegescorecard.ed.gov/) to find colleges and universities that are similar to Cal Poly.

In [307]:
df_college = pd.read_csv("https://datasci112.stanford.edu/data/college_attributes.csv")

df_college.set_index("Institution", inplace = True)

df_college

,City,State,AdmissionRate,Undergraduates,CarnegieClassification,Ownership,PCIP01,PCIP03,PCIP04,PCIP05,...,PCIP44,PCIP45,PCIP46,PCIP47,PCIP48,PCIP49,PCIP50,PCIP51,PCIP52,PCIP54
Institution,,,,,,,,,,,,,,,,,,,,,
Alabama A & M University,Normal,AL,0.7160,5098.0,Master's Colleges & Universities: Larger Programs,Public,0.0445,0.0071,0.0053,0.0000,...,0.0409,0.0249,0.0,0.0,0.0,0.0,0.0231,0.0000,0.1637,0.0000
University of Alabama at Birmingham,Birmingham,AL,0.8854,13284.0,Doctoral Universities: Very High Research Acti...,Public,0.0000,0.0000,0.0000,0.0020,...,0.0195,0.0239,0.0,0.0,0.0,0.0,0.0249,0.2088,0.2159,0.0141
University of Alabama in Huntsville,Huntsville,AL,0.7367,7358.0,Doctoral Universities: Very High Research Acti...,Public,0.0000,0.0000,0.0000,0.0000,...,0.0000,0.0127,0.0,0.0,0.0,0.0,0.0407,0.1341,0.1930,0.0073
Alabama State University,Montgomery,AL,0.9799,3495.0,Doctoral/Professional Universities,Public,0.0000,0.0000,0.0000,0.0000,...,0.0648,0.0196,0.0,0.0,0.0,0.0,0.0511,0.0904,0.1513,0.0059
The University of Alabama,Tuscaloosa,AL,0.7890,30725.0,Doctoral Universities: Very High Research Acti...,Public,0.0000,0.0061,0.0000,0.0019,...,0.0072,0.0661,0.0,0.0,0.0,0.0,0.0234,0.1077,0.2916,0.0096
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Florida Academy of Nursing,Miramar,FL,0.3088,239.0,Not applicable,Private for-profit,0.0000,0.0000,0.0000,0.0000,...,0.0000,0.0000,0.0,0.0,0.0,0.0,0.0000,1.0000,0.0000,0.0000
Herzing University-Tampa,Tampa,FL,0.9630,68.0,Not applicable,Private nonprofit,0.0000,0.0000,0.0000,0.0000,...,0.0000,0.0000,0.0,0.0,0.0,0.0,0.0000,0.0000,0.0000,0.0000
Abilene Christian University-Undergraduate Online,Addison,TX,1.0000,415.0,Not applicable,Private nonprofit,0.0000,0.0000,0.0000,0.0000,...,0.0000,0.0000,0.0,0.0,0.0,0.0,0.0000,0.0000,0.0000,0.0000


We'll want to single out Cal Poly, which we can do like this.

In [308]:
school_name = "California Polytechnic State University-San Luis Obispo"

cp = df_college.loc[school_name]

cp

City                                                        San Luis Obispo
State                                                                    CA
AdmissionRate                                                          0.33
Undergraduates                                                      21090.0
CarnegieClassification    Master's Colleges & Universities: Larger Programs
Ownership                                                            Public
PCIP01                                                               0.1084
PCIP03                                                               0.0255
PCIP04                                                               0.0441
PCIP05                                                               0.0019
PCIP09                                                               0.0353
PCIP10                                                               0.0175
PCIP11                                                               0.0326
PCIP12      

1\. Based on only the admission rate and the number of undergraduates, what schools are most similar to Cal Poly? Specify how you're making this decision.

In [309]:
college_quant = df_college[["AdmissionRate", "Undergraduates"]]

college_quant.head()

,AdmissionRate,Undergraduates
Institution,,
Alabama A & M University,0.7160,5098.0
University of Alabama at Birmingham,0.8854,13284.0
University of Alabama in Huntsville,0.7367,7358.0
Alabama State University,0.9799,3495.0
The University of Alabama,0.7890,30725.0


In [310]:
college_quant.isna().sum()

AdmissionRate     0
Undergraduates    0
dtype: int64

In [311]:
college_st = (college_quant - college_quant.mean()) / college_quant.std()

college_st.head()

,AdmissionRate,Undergraduates
Institution,,
Alabama A & M University,-0.071116,0.121332
University of Alabama at Birmingham,0.695587,1.175104
University of Alabama in Huntsville,0.022572,0.412258
Alabama State University,1.123294,-0.085020
The University of Alabama,0.259282,3.420258


In [312]:
cp_st = college_st.loc[school_name]

cp_st

AdmissionRate    -1.818149
Undergraduates    2.179959
Name: California Polytechnic State University-San Luis Obispo, dtype: float64

In [313]:
college_distance = np.sqrt(((college_st - cp_st) ** 2).sum(axis=1))

college_distance.head()

Institution
Alabama A & M University               2.700013
University of Alabama at Birmingham    2.707139
University of Alabama in Huntsville    2.552062
Alabama State University               3.712440
The University of Alabama              2.419517
dtype: float64

In [314]:
college_results = college_quant.assign(distance=college_distance)

college_results.head()

,AdmissionRate,Undergraduates,distance
Institution,,,
Alabama A & M University,0.7160,5098.0,2.700013
University of Alabama at Birmingham,0.8854,13284.0,2.707139
University of Alabama in Huntsville,0.7367,7358.0,2.552062
Alabama State University,0.9799,3495.0,3.712440
The University of Alabama,0.7890,30725.0,2.419517


In [315]:
college_results.drop(index=school_name).sort_values("distance").head()

,AdmissionRate,Undergraduates,distance
Institution,,,
University of California-Santa Barbara,0.2918,23081.0,0.309162
DeVry University-Illinois,0.4552,19729.0,0.593121
University of North Carolina at Chapel Hill,0.2040,19722.0,0.596846
Clemson University,0.4922,21577.0,0.736788
University of Virginia-Main Campus,0.2074,17041.0,0.761296


Answer: I standardized AdmissionRate and Undergraduates and calculated Euclidean distance from Cal Poly, excluding Cal Poly itself. The five closest schools were University of California-Santa Barbara, DeVry University-Illinois, University of North Carolina at Chapel Hill, Clemson University, and University of Virginia-Main Campus. University of California-Santa Barbara was the closest. These results describe similarity only in admission rate and undergraduate enrollment, not overall similarity between the schools.

2\. Now consider the admission rate, the number of undergraduates, and also the [Carnegie classification](https://en.wikipedia.org/wiki/Carnegie_Classification_of_Institutions_of_Higher_Education) of the type of school, and the ownership (public, private, etc.) Based on these variables, what schools are most similar to Cal Poly? Specify how you're making this decision.

In [316]:
college_categories = pd.get_dummies(df_college[["CarnegieClassification", "Ownership"]], dtype=int)

college_categories.head()

,CarnegieClassification_Associate's Colleges: High Career & Technical-High Nontraditional,CarnegieClassification_Associate's Colleges: High Career & Technical-High Traditional,CarnegieClassification_Associate's Colleges: High Career & Technical-Mixed Traditional/Nontraditional,CarnegieClassification_Associate's Colleges: High Transfer-High Nontraditional,CarnegieClassification_Associate's Colleges: High Transfer-High Traditional,CarnegieClassification_Associate's Colleges: High Transfer-Mixed Traditional/Nontraditional,CarnegieClassification_Baccalaureate Colleges: Arts & Sciences Focus,CarnegieClassification_Baccalaureate Colleges: Diverse Fields,CarnegieClassification_Baccalaureate/Associate's Colleges: Associate's Dominant,CarnegieClassification_Baccalaureate/Associate's Colleges: Mixed Baccalaureate/Associate's,...,CarnegieClassification_Special Focus Four-Year: Other Special Focus Institutions,CarnegieClassification_Special Focus Four-Year: Research Institution,CarnegieClassification_Special Focus Two-Year: Arts & Design,CarnegieClassification_Special Focus Two-Year: Health Professions,CarnegieClassification_Special Focus Two-Year: Other Fields,CarnegieClassification_Special Focus Two-Year: Technical Professions,CarnegieClassification_Tribal Colleges,Ownership_Private for-profit,Ownership_Private nonprofit,Ownership_Public
Institution,,,,,,,,,,,,,,,,,,,,,
Alabama A & M University,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
University of Alabama at Birmingham,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
University of Alabama in Huntsville,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
Alabama State University,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
The University of Alabama,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1


In [317]:
college_mixed = pd.concat([college_st, college_categories], axis=1)

college_mixed.head()

,AdmissionRate,Undergraduates,CarnegieClassification_Associate's Colleges: High Career & Technical-High Nontraditional,CarnegieClassification_Associate's Colleges: High Career & Technical-High Traditional,CarnegieClassification_Associate's Colleges: High Career & Technical-Mixed Traditional/Nontraditional,CarnegieClassification_Associate's Colleges: High Transfer-High Nontraditional,CarnegieClassification_Associate's Colleges: High Transfer-High Traditional,CarnegieClassification_Associate's Colleges: High Transfer-Mixed Traditional/Nontraditional,CarnegieClassification_Baccalaureate Colleges: Arts & Sciences Focus,CarnegieClassification_Baccalaureate Colleges: Diverse Fields,...,CarnegieClassification_Special Focus Four-Year: Other Special Focus Institutions,CarnegieClassification_Special Focus Four-Year: Research Institution,CarnegieClassification_Special Focus Two-Year: Arts & Design,CarnegieClassification_Special Focus Two-Year: Health Professions,CarnegieClassification_Special Focus Two-Year: Other Fields,CarnegieClassification_Special Focus Two-Year: Technical Professions,CarnegieClassification_Tribal Colleges,Ownership_Private for-profit,Ownership_Private nonprofit,Ownership_Public
Institution,,,,,,,,,,,,,,,,,,,,,
Alabama A & M University,-0.071116,0.121332,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
University of Alabama at Birmingham,0.695587,1.175104,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
University of Alabama in Huntsville,0.022572,0.412258,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
Alabama State University,1.123294,-0.085020,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
The University of Alabama,0.259282,3.420258,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1


In [318]:
cp_mixed = college_mixed.loc[school_name]

cp_mixed

AdmissionRate                                                                                           -1.818149
Undergraduates                                                                                           2.179959
CarnegieClassification_Associate's Colleges: High Career & Technical-High Nontraditional                 0.000000
CarnegieClassification_Associate's Colleges: High Career & Technical-High Traditional                    0.000000
CarnegieClassification_Associate's Colleges: High Career & Technical-Mixed Traditional/Nontraditional    0.000000
CarnegieClassification_Associate's Colleges: High Transfer-High Nontraditional                           0.000000
CarnegieClassification_Associate's Colleges: High Transfer-High Traditional                              0.000000
CarnegieClassification_Associate's Colleges: High Transfer-Mixed Traditional/Nontraditional              0.000000
CarnegieClassification_Baccalaureate Colleges: Arts & Sciences Focus                    

In [319]:
college_distance_mixed = np.sqrt(((college_mixed - cp_mixed) ** 2).sum(axis=1))

college_distance_mixed.head()

Institution
Alabama A & M University               2.700013
University of Alabama at Birmingham    3.054276
University of Alabama in Huntsville    2.917708
Alabama State University               3.972683
The University of Alabama              2.802510
dtype: float64

In [320]:
college_results_mixed = df_college.assign(distance=college_distance_mixed)

college_results_mixed.head()

,City,State,AdmissionRate,Undergraduates,CarnegieClassification,Ownership,PCIP01,PCIP03,PCIP04,PCIP05,...,PCIP45,PCIP46,PCIP47,PCIP48,PCIP49,PCIP50,PCIP51,PCIP52,PCIP54,distance
Institution,,,,,,,,,,,,,,,,,,,,,
Alabama A & M University,Normal,AL,0.7160,5098.0,Master's Colleges & Universities: Larger Programs,Public,0.0445,0.0071,0.0053,0.0000,...,0.0249,0.0,0.0,0.0,0.0,0.0231,0.0000,0.1637,0.0000,2.700013
University of Alabama at Birmingham,Birmingham,AL,0.8854,13284.0,Doctoral Universities: Very High Research Acti...,Public,0.0000,0.0000,0.0000,0.0020,...,0.0239,0.0,0.0,0.0,0.0,0.0249,0.2088,0.2159,0.0141,3.054276
University of Alabama in Huntsville,Huntsville,AL,0.7367,7358.0,Doctoral Universities: Very High Research Acti...,Public,0.0000,0.0000,0.0000,0.0000,...,0.0127,0.0,0.0,0.0,0.0,0.0407,0.1341,0.1930,0.0073,2.917708
Alabama State University,Montgomery,AL,0.9799,3495.0,Doctoral/Professional Universities,Public,0.0000,0.0000,0.0000,0.0000,...,0.0196,0.0,0.0,0.0,0.0,0.0511,0.0904,0.1513,0.0059,3.972683
The University of Alabama,Tuscaloosa,AL,0.7890,30725.0,Doctoral Universities: Very High Research Acti...,Public,0.0000,0.0061,0.0000,0.0019,...,0.0661,0.0,0.0,0.0,0.0,0.0234,0.1077,0.2916,0.0096,2.802510


In [321]:
closest_colleges_mixed = college_results_mixed.drop(index=school_name).sort_values("distance")

closest_colleges_mixed[["AdmissionRate", "Undergraduates", "CarnegieClassification", "Ownership", "distance"]].head()

,AdmissionRate,Undergraduates,CarnegieClassification,Ownership,distance
Institution,,,,,
CUNY Hunter College,0.4590,17293.0,Master's Colleges & Universities: Larger Programs,Public,0.761441
CUNY Bernard M Baruch College,0.5056,15483.0,Master's Colleges & Universities: Larger Programs,Public,1.073601
CUNY John Jay College of Criminal Justice,0.4458,12834.0,Master's Colleges & Universities: Larger Programs,Public,1.184989
CUNY Brooklyn College,0.5136,12567.0,Master's Colleges & Universities: Larger Programs,Public,1.376321
University of California-Santa Barbara,0.2918,23081.0,Doctoral Universities: Very High Research Acti...,Public,1.447612


Answer: I standardized AdmissionRate and Undergraduates and converted CarnegieClassification and Ownership into dummy variables without scaling them. I calculated Euclidean distance from Cal Poly and excluded Cal Poly itself.

The five closest schools were CUNY Hunter College, CUNY Bernard M Baruch College, CUNY John Jay College of Criminal Justice, CUNY Brooklyn College, and University of California Santa Barbara. CUNY Hunter College was the closest. In this dataset, all five are public, and the first four have the same Carnegie classification as Cal Poly. Adding the categorical variables changed the rankings from part 1.

3\. The columns whose names begin with "PCIP" contain the proportions of students at each school studying various fields (e.g., Engineering, Psychology). Each field is represented by a two-digit code called the [CIP code](https://nces.ed.gov/ipeds/cipcode/browse.aspx?y=55).

If we only consider the proportions of students studying various fields, what schools are most similar to Cal Poly? Specify how you're making this decision.

In [322]:
college_pcip = df_college.filter(regex="^PCIP")

college_pcip.head()

,PCIP01,PCIP03,PCIP04,PCIP05,PCIP09,PCIP10,PCIP11,PCIP12,PCIP13,PCIP14,...,PCIP44,PCIP45,PCIP46,PCIP47,PCIP48,PCIP49,PCIP50,PCIP51,PCIP52,PCIP54
Institution,,,,,,,,,,,,,,,,,,,,,
Alabama A & M University,0.0445,0.0071,0.0053,0.0000,0.0000,0.0285,0.0658,0.0,0.0391,0.1210,...,0.0409,0.0249,0.0,0.0,0.0,0.0,0.0231,0.0000,0.1637,0.0000
University of Alabama at Birmingham,0.0000,0.0000,0.0000,0.0020,0.0333,0.0000,0.0229,0.0,0.0667,0.0559,...,0.0195,0.0239,0.0,0.0,0.0,0.0,0.0249,0.2088,0.2159,0.0141
University of Alabama in Huntsville,0.0000,0.0000,0.0000,0.0000,0.0140,0.0000,0.0692,0.0,0.0218,0.3028,...,0.0000,0.0127,0.0,0.0,0.0,0.0,0.0407,0.1341,0.1930,0.0073
Alabama State University,0.0000,0.0000,0.0000,0.0000,0.0923,0.0000,0.0530,0.0,0.0589,0.0138,...,0.0648,0.0196,0.0,0.0,0.0,0.0,0.0511,0.0904,0.1513,0.0059
The University of Alabama,0.0000,0.0061,0.0000,0.0019,0.0877,0.0000,0.0148,0.0,0.0262,0.1198,...,0.0072,0.0661,0.0,0.0,0.0,0.0,0.0234,0.1077,0.2916,0.0096


In [323]:
college_pcip.isna().sum()

PCIP01    0
PCIP03    0
PCIP04    0
PCIP05    0
PCIP09    0
PCIP10    0
PCIP11    0
PCIP12    0
PCIP13    0
PCIP14    0
PCIP15    0
PCIP16    0
PCIP19    0
PCIP22    0
PCIP23    0
PCIP24    0
PCIP25    0
PCIP26    0
PCIP27    0
PCIP29    0
PCIP30    0
PCIP31    0
PCIP38    0
PCIP39    0
PCIP40    0
PCIP41    0
PCIP42    0
PCIP43    0
PCIP44    0
PCIP45    0
PCIP46    0
PCIP47    0
PCIP48    0
PCIP49    0
PCIP50    0
PCIP51    0
PCIP52    0
PCIP54    0
dtype: int64

In [324]:
cp_pcip = college_pcip.loc[school_name]

cp_pcip

PCIP01    0.1084
PCIP03    0.0255
PCIP04    0.0441
PCIP05    0.0019
PCIP09    0.0353
PCIP10    0.0175
PCIP11    0.0326
PCIP12    0.0000
PCIP13    0.0130
PCIP14    0.2314
PCIP15    0.0110
PCIP16    0.0019
PCIP19    0.0000
PCIP22    0.0000
PCIP23    0.0147
PCIP24    0.0229
PCIP25    0.0000
PCIP26    0.0495
PCIP27    0.0173
PCIP29    0.0000
PCIP30    0.0223
PCIP31    0.0419
PCIP38    0.0073
PCIP39    0.0000
PCIP40    0.0205
PCIP41    0.0000
PCIP42    0.0288
PCIP43    0.0000
PCIP44    0.0000
PCIP45    0.0588
PCIP46    0.0000
PCIP47    0.0000
PCIP48    0.0000
PCIP49    0.0000
PCIP50    0.0130
PCIP51    0.0060
PCIP52    0.1637
PCIP54    0.0110
Name: California Polytechnic State University-San Luis Obispo, dtype: float64

In [325]:
college_distance_pcip = np.sqrt(((college_pcip - cp_pcip) ** 2).sum(axis=1))

college_distance_pcip.head()

Institution
Alabama A & M University               0.216428
University of Alabama at Birmingham    0.322242
University of Alabama in Huntsville    0.208100
Alabama State University               0.321949
The University of Alabama              0.253177
dtype: float64

In [326]:
closest_colleges_pcip = college_distance_pcip.drop(index=school_name).sort_values()

closest_colleges_pcip.head()

Institution
North Carolina State University at Raleigh    0.084343
Iowa State University                         0.085054
University of Illinois Urbana-Champaign       0.111475
Mississippi State University                  0.118799
Texas A & M University-College Station        0.123973
dtype: float64

Answer: I used only the columns beginning with PCIP and calculated Euclidean distance from Cal Poly, excluding Cal Poly itself. I used the original proportions without standardizing them because all these variables are measured on the same scale.

The five closest schools were North Carolina State University at Raleigh, Iowa State University, University of Illinois Urbana-Champaign, Mississippi State University, and Texas A & M University College Station. North Carolina State University at Raleigh was the closest. These results describe similarity in the proportions of students studying different fields, not overall similarity between the schools.